# `01_langgraph_basic.ipynb`

## Core Concept
1. `state` -> 노드들을 관통하는 데이터
2. Langgrpah 기획 -> 노드/엣지 그림 + State에 어떤 데이터를 담을지를 기획하는 것

### 예시 - 여행 계획 세워주는 Workflow
1. wf가 채워야 하는 정보
    - 목적지: `str`
    - 여행일수: `int`
    - 목적지에서 방문할 장소들: `list[str]`
2. Node -Edge
    1. 목적지 채우는 Node
    2. 여행일수 Node
    3. 방문할 장소 채우는 Node
    4. 최종 답변을 만들어주는 Node

In [2]:
from dotenv import load_dotenv
from typing_extensions import TypedDict, List
from langgraph.graph import StateGraph , START, END
from langchain.chat_models import init_chat_model


load_dotenv()

True

In [ ]:
# State 정의
class TripState(TypedDict): # TypedDict? 딕셔너리(k-v)인데, Value의 자료형을 미리 정해놓은 dict
    message: str        # 사용자 입력메세지 - 시작과 동시에 채워질 데이터
    destination: str    # 목적지
    days: int           # 여행일수
    places: List[str]   # 방문장소목록 ex. ['경복궁', 'N타워', '한강공원']   -> List[str]과 list[str]의 차이점: List[str] 에서 안에 str이 아닌게 들어가면 경고가 나옴 오류는 x (TypedDict import 된 상태에서만)
    answer: str         # 최종 답변 - 종료시 완성될 메세지

In [8]:
# Node 정의

def set_destination_node(state: TripState):
  import random #함수안에 import는 안좋음 그냥 지금은 예시니까
  pick = random.choice(['서울', '부산', '제주'])
  # State 갱신은 Langgrapth가 Stete 바뀐 부분만 넘기면 자동으로 갱신해서 넘김(하나씩 내리는데 갯수가 많으면 일일이 다 해야되니까)
  return {
      #'message' : state['message'], -> 그래서 이거 작성 안해도됨
      'destination': pick
  }
  

def set_days_node(state: TripState):
    return {'days': 3}

def collect_places_node(state: TripState): 
    destination = state['destination']
    if destination == '서울':
        places = ['광화문', '국중박', '한강', 'N타워']
    elif destination == '부산':
        places = ['해운대', '광안리', '서면', '기장']
    elif destination == '제주' :
        places = ['중문', '우도', '협재']
    return {'places': places}

def make_answer_node(state: TripState):
    destination = state['destination']
    days = state['days']
    places = state['places']

    llm = init_chat_model('openai:gpt-4.1-mini') # llm 노드 추가

    ## 방법1 - dict이용
    aimessage = llm.invoke([
        {'role': 'system', 'content': '너는 여행계획을 짜주는 어시스턴트야'},
        {'role': 'user', 'content': '서울 여행 가고싶다.'}
    ])

    ## 방법2 - Human, System, AI 메세지 이용 -> import langchain.messages 필요
    from langchain.messages import HumanMessage, SystemMessage, AIMessage ## 실제로는 맨위에 작성 import기 때문에
    llm.invoke([ 
        SystemMessage(content=f'너는 여행계획을 짜주는 어시스턴스야'),
        HumanMessage(content=f'{destination}에 {days}일동안 {places}를 갈거야.')
    ])


    answer = f'''{destination}을 추천합니다. {days}박 코스로 가시면 좋아요. {places}는 꼭 가보세요'''
    return {'answer': aimessage.content}

In [9]:
# builder 에서 Node들 연결 = 조립
builder = StateGraph(TripState) # TripState를 공유하는 Grapth 빌더

# Node 등록(이름, 노드(함수))
builder.add_node('set_destination_node',set_destination_node)
builder.add_node('collect_places_node', collect_places_node)
builder.add_node('set_days_node', set_days_node)
builder.add_node('make_answer_node', make_answer_node)

# Edge 연결 - 노드끼리 연결 -> 순서중요
builder.add_edge(START, 'set_destination_node')
builder.add_edge('set_destination_node', 'set_days_node')
builder.add_edge('set_days_node', 'collect_places_node')
builder.add_edge('collect_places_node', 'make_answer_node')
builder.add_edge('make_answer_node', END)
# graph 컴파일 (실행 가능하게 만든다)
graph = builder.compile()

In [10]:
#Langchain 세상에서 "실행"은 .invoke()가 기본 메서드
graph.invoke({  # Typed DICT니까, 
    'message' : '나 여행가고 싶어' # 최초에 채울 k - v
 })

{'message': '나 여행가고 싶어',
 'destination': '제주',
 'days': 3,
 'places': ['중문', '우도', '협재'],
 'answer': '서울 여행을 계획하신다니 좋네요! 여행 기간은 얼마나 되시나요? 그리고 어떤 스타일의 여행을 원하시나요? 예를 들어, 맛집 탐방, 문화 명소 방문, 쇼핑, 자연 경관 감상 등 선호하는 활동을 알려주시면 맞춤 일정을 추천해드릴게요.'}